# AQNG vs Full-QNG — GitHub launcher
هاد النوتبوك كيجبد آخر نسخة من benchmark مباشرة من GitHub. غير بدّل الhyperparameters فـCONFIG وخدم الخلايا.


In [ ]:
!pip -q install "pennylane>=0.45,<0.46" "pennylane-lightning>=0.45,<0.46" scikit-learn pandas matplotlib tqdm


## Hyperparameters — تحكم فيهم هنا


In [ ]:
CONFIG = dict(
    seeds=[0, 1],
    steps=8,
    fixed_lr=0.03,
    n_samples=40,
    loss_batch=8,
    metric_batch=2,
    metric_every=2,
    lam=1e-3,
    n_qubits=4,
    n_layers=2,
    show_plot=True,
    verbose=True,
)

GITHUB_BRANCH = "main"

# Folder name under results/generic_u1/ on GitHub
RESULTS_TAG = "fixed_lr_run_001"


## Fetch latest benchmark code from GitHub
كل مرة كتخدم هاد الخلية كتجيب آخر نسخة من `main`.


In [ ]:
import importlib.util
import urllib.request
from pathlib import Path

RAW_URL = (
    "https://raw.githubusercontent.com/"
    f"AHDMarwan/aqng/{GITHUB_BRANCH}/experiments/generic_u1_fixed_lr.py"
)
LOCAL_FILE = Path("/content/generic_u1_fixed_lr.py")

urllib.request.urlretrieve(RAW_URL, LOCAL_FILE)
print("Fetched:", RAW_URL)

spec = importlib.util.spec_from_file_location("generic_u1_bench", LOCAL_FILE)
bench = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bench)


## Run


In [ ]:
results, summary, paired, curves = bench.run_benchmark(**CONFIG)

display(results)
display(summary)
if len(paired):
    display(
        paired[
            ["vqc", "seed",
             "delta_loss_AQNG_minus_QNG",
             "speedup_QNG_over_AQNG"]
        ]
    )


## Push results directly to GitHub

1. In Colab: **Secrets** → add a secret named `GITHUB_TOKEN`.
2. Use a fine-grained GitHub token with **Contents: Read and write** permission for `AHDMarwan/aqng`.
3. Change `RESULTS_TAG` above for each experiment you want to keep separately.

The token is read from Colab Secrets and is never written into the notebook/results.


In [ ]:
import base64
import json
from datetime import datetime, timezone

import requests
import pennylane as qml
import sklearn
from google.colab import userdata

GITHUB_REPO = "AHDMarwan/aqng"
GITHUB_BRANCH = "main"
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError(
        "Add a Colab Secret named GITHUB_TOKEN first "
        "(fine-grained token, Contents: read/write for AHDMarwan/aqng)."
    )

API_ROOT = f"https://api.github.com/repos/{GITHUB_REPO}"
HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "X-GitHub-Api-Version": "2022-11-28",
}

def github_file_metadata(path):
    r = requests.get(
        f"{API_ROOT}/contents/{path}",
        headers=HEADERS,
        params={"ref": GITHUB_BRANCH},
        timeout=30,
    )
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()

def github_put_text(path, text, message):
    """Create or replace one UTF-8 text file on GitHub."""
    meta = github_file_metadata(path)

    payload = {
        "message": message,
        "content": base64.b64encode(text.encode("utf-8")).decode("ascii"),
        "branch": GITHUB_BRANCH,
    }
    if meta is not None:
        payload["sha"] = meta["sha"]

    r = requests.put(
        f"{API_ROOT}/contents/{path}",
        headers=HEADERS,
        json=payload,
        timeout=60,
    )
    if r.status_code not in (200, 201):
        raise RuntimeError(
            f"GitHub upload failed for {path}: "
            f"{r.status_code} {r.text[:500]}"
        )
    return r.json()

# Record the exact benchmark source blob used by this run.
source_meta = github_file_metadata(
    "experiments/generic_u1_fixed_lr.py"
)
source_blob_sha = source_meta["sha"] if source_meta else None

# Convert curves dict to a normal table.
curve_rows = []
for (vqc, method, seed), curve in curves.items():
    for step, loss in enumerate(curve, 1):
        curve_rows.append({
            "vqc": vqc,
            "method": method,
            "seed": seed,
            "step": step,
            "train_loss_before": float(loss),
        })
curves_df = pd.DataFrame(curve_rows)

run_meta = {
    "results_tag": RESULTS_TAG,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "repository": GITHUB_REPO,
    "branch": GITHUB_BRANCH,
    "benchmark_path": "experiments/generic_u1_fixed_lr.py",
    "benchmark_blob_sha": source_blob_sha,
    "config": CONFIG,
    "versions": {
        "pennylane": qml.__version__,
        "sklearn": sklearn.__version__,
        "pandas": pd.__version__,
    },
}

base = f"results/generic_u1/{RESULTS_TAG}"

files_to_push = {
    f"{base}/results.csv": results.to_csv(index=False),
    f"{base}/summary.csv": summary.to_csv(index=False),
    f"{base}/paired.csv": paired.to_csv(index=False),
    f"{base}/curves.csv": curves_df.to_csv(index=False),
    f"{base}/config.json": json.dumps(
        run_meta, indent=2, sort_keys=True
    ),
}

# GitHub Contents API writes are intentionally serial.
for path, text in files_to_push.items():
    print("Uploading:", path)
    github_put_text(
        path,
        text,
        message=f"Add AQNG benchmark results: {RESULTS_TAG}",
    )

print(
    f"Done: https://github.com/{GITHUB_REPO}/tree/"
    f"{GITHUB_BRANCH}/{base}"
)
